## **Mini-Batch Gradient Descent**

### **Topic Roadmap**

 **1. Imports & Dataset Preparation**

 **2. OLS Baseline Model**

 **3. Custom Mini-Batch Gradient Descent**

 **4. Scikit-Learn Implementation (`partial_fit`)**

 **5. Key Revision Notes**

### **1. Imports & Dataset Preparation**

Import the required libraries, load the diabetes regression dataset, and split it into training and testing sets[cite: 5].

In [1]:
import random
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Load diabetes dataset
X, y = load_diabetes(return_X_y=True)

# Create training and testing splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

### **2. OLS Baseline Model**

Train a standard Ordinary Least Squares (OLS) Linear Regression model to establish a baseline $R^2$ score[cite: 5].

In [2]:
# Initialize and train the baseline model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Evaluate baseline performance
y_pred_ols = lr_model.predict(X_test)
baseline_r2 = r2_score(y_test, y_pred_ols)

print(f"Baseline OLS R2 Score: {baseline_r2:.4f}")

Baseline OLS R2 Score: 0.4399


### **3. Custom Mini-Batch Gradient Descent**

Build a custom Mini-Batch Gradient Descent regressor from scratch.

Mini-Batch Gradient Descent strikes a balance between Batch Gradient Descent and Stochastic Gradient Descent by computing parameter updates on small, random subsets (`batch_size`) of the data instead of single samples or the entire dataset[cite: 5].

In [3]:
class MBGDRegressor:
    def __init__(self, batch_size, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        
    def fit(self, X_train, y_train):
        # Initialize intercept to 0 and coefficients to an array of ones
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        
        num_batches = int(X_train.shape[0] / self.batch_size)
        
        # Iterate over the specified number of epochs
        for i in range(self.epochs):
            # Process the data in mini-batches
            for j in range(num_batches):
                # Randomly sample a mini-batch of indices
                idx = random.sample(range(X_train.shape[0]), self.batch_size)
                
                # Calculate prediction for the mini-batch
                y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_
                
                # Calculate loss gradients over the mini-batch
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                coef_der = -2 * np.dot((y_train[idx] - y_hat), X_train[idx])

                # Apply parameter updates
                self.intercept_ -= (self.lr * intercept_der)
                self.coef_ -= (self.lr * coef_der)
                
    def predict(self, X_test):
        # Calculate final predictions using the learned parameters
        return np.dot(X_test, self.coef_) + self.intercept_

# Initialize and train the custom model
batch_size = int(X_train.shape[0] / 50)
mbgd_model = MBGDRegressor(batch_size=batch_size, learning_rate=0.01, epochs=100)
mbgd_model.fit(X_train, y_train)

# Evaluate custom model performance
y_pred_mbgd = mbgd_model.predict(X_test)
mbgd_r2 = r2_score(y_test, y_pred_mbgd)

print(f"\nCustom Mini-Batch GD R2 Score: {mbgd_r2:.4f}")


Custom Mini-Batch GD R2 Score: 0.4525


### **4. Scikit-Learn Implementation (`partial_fit`)**

Implement Mini-Batch Gradient Descent using Scikit-Learn's `SGDRegressor`. Instead of using `.fit()`, which runs standard SGD, manually feed mini-batches iteratively using the `.partial_fit()` method[cite: 5].

In [4]:
# Initialize the scikit-learn SGD model
sklearn_sgd = SGDRegressor(learning_rate='constant', eta0=0.1)

batch_size_sk = 35

# Train iteratively using partial_fit to simulate mini-batch updates
for i in range(100):
    idx = random.sample(range(X_train.shape[0]), batch_size_sk)
    sklearn_sgd.partial_fit(X_train[idx], y_train[idx])

# Evaluate sklearn model performance
y_pred_sklearn = sklearn_sgd.predict(X_test)
sklearn_r2 = r2_score(y_test, y_pred_sklearn)

print(f"Sklearn Mini-Batch SGD R2 Score: {sklearn_r2:.4f}")

Sklearn Mini-Batch SGD R2 Score: 0.4257


### **Key Revision Notes**

- **Mini-Batch Gradient Descent:** Updates parameters using a small, randomized subset (mini-batch) of the dataset per iteration, serving as a compromise between the robustness of Batch GD and the high-speed convergence of Stochastic GD[cite: 5].
- **Batch Size:** A crucial hyperparameter (commonly set between 32 and 512). A batch size that is too small mimics the erratic convergence of pure SGD, while one that is too large slows down the step updates similar to Batch GD.
- **`partial_fit` Method:** In Scikit-Learn, models like `SGDRegressor` support incremental learning via `.partial_fit(X_batch, y_batch)`. This allows out-of-core learning and custom mini-batch loops without retaining the full dataset in memory[cite: 5].
- **Vectorization Advantage:** Mini-batch GD exploits highly optimized matrix operations (`np.dot`) across the batch instances, offering significantly faster computation than processing samples one by one as done in standard SGD[cite: 5].